# Code Notebook for Data 275P Homework 3

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Imports to use for optimization
from functools import partial
import scipy.optimize

import requests                                      # reading data
from io import StringIO

<hr style="height: 3px;">

### Load the data

In [ ]:
url = 'https://www.ics.uci.edu/~ihler/classes/data275P/data/data275P-hw1-gamma.txt'
nTrain = 15216

with requests.get(url) as link:
    datafile = StringIO(link.text)
    gamma_data = np.genfromtxt(datafile,delimiter=' ')
    nTrain = int( gamma_data.shape[0] * 0.8 )
    train, train_labels = gamma_data[:nTrain,:-1], gamma_data[:nTrain,-1]
    test, test_labels = gamma_data[nTrain:,:-1], gamma_data[nTrain:,-1]

# train: training matrix of 10 continuous observations from star showers.
# train_labels: column vector of {0,1} training labels where 0=hadron,1=gamma.
# test: test matrix of 10 continuous observations from star showers
# test_labels: column vector of {0,1} test labels where 0=hadron,1=gamma.

<hr style="height: 3px;">

### Helper function for Logistic Regression

In [ ]:
def generate_features(x, feature_type):
    '''Function to generate feature mappings on the gamma dataset.
        Arguments:
            x: data features (either training or testing data) (NxD)
            feature_type [string]: feature type to generate, {'linear','diag_quad','full_quad'}
        
        Returns:
            phi: New feature data (either: Nx[D+1], Nx[2D+1], or Nx[(D+1)D/2+D+1])
    '''
    num_features = x.shape[-1]
    if feature_type.lower()=='linear':
        # constant bias plus original features
        phi = np.zeros((x.shape[0],num_features+1))
        phi[:, 0] = 1.0
        phi[:, 1:] = x
    elif feature_type.lower()=='diag_quad':
        # "diagonal" quadratic: linear features plus x_i^2 for each feature i
        phi = np.zeros((x.shape[0], (num_features*2)+1))
        phi[:, 0] = 1.0
        phi[:, 1:(num_features+1)] = x
        phi[:, (num_features+1):] = x**2
    elif feature_type.lower()=='full_quad':  
        # All quadratic features; use same order as diag_quad, and append all xi*xj i!=j
        phi = np.zeros((x.shape[0], (num_features+1)*num_features//2+num_features+1))
        phi[:, 0] = 1.0
        phi[:, 1:(num_features+1)] = x
        phi[:, (num_features+1):2*num_features+1] = x**2
        ij = 2*num_features+1
        for i in range(num_features):
            for j in range(i+1,num_features):
                phi[:,ij] = x[:,i]*x[:,j]
                ij += 1
    else:
        raise ValueError(f'Unknown feature type {feature_type}; must be one of ["linear","diag_quad","full_quad"]')
    return phi

You will probably also want to normalize the features before you use them; this is not absolutely necessary for the MAP estimation (though it can speed it up), but will definitely be necessary for the MCMC estimates.

In [ ]:
def normalize_features(phi_train,phi_test):
    '''Rescale phi_train and phi_test to the range [-1,1]'''
    shift = (np.max(phi_train,axis=0,keepdims=True)-np.min(phi_train,axis=0,keepdims=True))/2;
    scale = 1.*shift; 
    shift[scale==0]=0.; scale[scale==0]=1.;  # avoid rescaling constant features
    phi_train = (phi_train - shift)/scale
    phi_test = (phi_test - shift)/scale
    return phi_train,phi_test

<hr style="height: 3px;">

### Your functions (to be completed)
Update/complete the code in these functions for your assignment. Don't change the function names or signatures, as these will be used by the autograder.

In [6]:
def loss_func_logistic(w, x, t, k, alpha):
    ''' Compute the L2-regularized logistic loss function 

    Arguments:
        w: estimated feature weight vector
        x: model matrix (NxM)
        t: response vector (Nx1)
        k: number of response classes
        alpha: regularization constant
        
    Returns:
        The loss function value
    '''
    try:    w = w.reshape( (x.shape[1],k) )    # w may be passed in as a vector
    except: raise ValueError(f'Could not convert weights {w.shape} to (N,K)!')

    loss = ...    # TODO: compute the loss
    return loss

In [7]:
def gradient_loss_func_logistic(w, x, t, k, alpha):
    ''' Compute the gradient of the L2-regularized logistic loss (with respect to w) 

    Arguments:
        w: estimated feature weight vector
        x: model matrix (NxM)
        t: response vector (Nx1)
        k: number of response classes
        alpha: regularization constant

    Returns:
        The gradient of loss function with respect to w
    '''
    try:    w = w.reshape( (x.shape[1],k) )    # w may be passed in as a vector
    except: raise ValueError(f'Could not convert weights {w.shape} to (M,K)!')

    gradient = ...    # TODO: compute the gradient
    
    gradient = gradient.flatten()  # need to return dw as a vector
    return gradient

In [ ]:
# Example options for scipy.optimize:
options = {'maxiter': 1000, 'ftol':1e-7}

In [9]:
def gamma_logistic_regression_model(phi_train, phi_test, train_labels, test_labels, options, alpha=1e-6):
    ''' Train a logistic regression model on gamma dataset
    
    Arguments:
        phi_train: training dataset (NxM)
        phi_test: testing dataset (N'xM)
        train_labels: vector of training labels (Nx1)
        test_labels: vector of testing labels (N'x1)
        options: Dictionary object for scipy.optimize.minimize (dict())
        alpha: alpha value used in logistic loss and gradient functions (float)
        
        (N: number of elements in training data, N': number of elements in testing data)
    Returns:
        train_accuracy: (float)
        train_loss: (float)
        test_accuracy: (float)
        test_loss: (float)
    '''
    # Any setup you may need
    k = ... # number of classes in the classifier data 

    # Create the partial functions that we will be using for the optimization.
    # These functions bind the loss and gradient functions such that we can just 
    # pass in a value for w and evaluate the function.
    # You must derive and implement the gradient function, NOT approximate numerically.
    loss_func = partial(loss_func_logistic, \
        x=phi_train, t=train_labels, k=k, alpha=alpha)
    grad_func = partial(gradient_loss_func_logistic, \
        x=phi_train, t=train_labels, k=k, alpha=alpha)

    # Now train the model using e.g. scipy.optimize.minimize
    # TODO ...
    
    train_loss = ...
    test_loss = ...

    train_accuracy = ...
    test_accuracy = ...
	
    return train_accuracy, train_loss, test_accuracy, test_loss

<hr style="height: 3px;">

### Code for your PDF submission

<hr style="height: 3px;">

### Pyro and MCMC
Use this section's example code to complete the last question, using MCMC to provide Bayesian predictive probabilities from your logistic regression model.

Since this section is slow, use "#!no-autograder" to avoid running the cells when you submit.

In [ ]:
#!pip install pyro-ppl  ## you may need to install pyro

In [ ]:
#!no-autograder
import torch
import pyro
import pyro.distributions as dist

In [ ]:
#!no-autograder
sigma = lambda z: 1./(1+torch.exp(-z))    # useful later to have a logistic function defined

def model(x,y):
    D = x.shape[1]
    w = pyro.sample( 'w', dist.Normal( torch.zeros(D), torch.ones(D) ) ) 
    with pyro.plate("data", len(x)):
        responses = (x @ w)  ## assumes constant term in features (!)
        pyro.sample("Y",dist.Bernoulli(logits=responses),obs=y)

In [ ]:
#!no-autograder
phi_train = generate_features(train,'linear')
phi_test = generate_features(test,'linear')

# For the MCMC run, it's important that the features be normalized somehow first:
phi_train,phi_test = normalize_features(phi_train,phi_test)

# and we have to convert everything to torch tensor types
phi_train = torch.Tensor(phi_train)
phi_test = torch.Tensor(phi_test)
y_train = torch.Tensor(train_labels)

In [ ]:
#!no-autograder

# This may take a few minutes to run...
nuts_kernel = pyro.infer.NUTS(model, jit_compile=False)
mcmc = pyro.infer.MCMC(nuts_kernel, num_samples=200, warmup_steps=100, num_chains=1)
mcmc.run(phi_train,y_train)

In [ ]:
#!no-autograder
samples = mcmc.get_samples()['w']
per_sample = torch.zeros(phi_test.shape[0], len(samples))
for i in range(len(samples)):
    per_sample[:,i] = sigma( phi_test @ samples[i] ) #.reshape(-1,1) )

yhat_test = per_sample.mean(1)  ## average predicted probability across posterior samples